# Level 5 — Data Mining Challenge: *The 1,000-Pick*

**규칙**: Set B (이미지 + 라벨 공개) 에서 **최대 1,000장** 을 선택하여 학습 셋에 추가하고, best 모델을 다시 학습하세요.

> **Set B 의 라벨이 공개되어 있다는 점에 주의**하세요. 본 Level 의 평가 본질은 "*주어진 풀에서 어떤 1,000장이 가장 가치 있는가*" — 즉, 라벨을 알고 있다고 가정한 상태에서의 효율적인 부분집합 선택입니다.

**본 PA에서 가장 큰 비중 (25%)** 을 차지하는 Level 입니다. 어떤 *알고리즘* 으로 1,000장을 골랐는지 — 그 *근거* — 가 변별력의 본진입니다. Curation Report 로 정리합니다.

채점 메트릭:
$$\text{DI} = \frac{\text{Avg-MF1}(\text{본인 picks}) - \text{Avg-MF1}(\text{random picks})}{\text{Avg-MF1}(\text{random picks})}$$

## 검토해 볼 만한 전략

| 전략 | 핵심 아이디어 | Set B 라벨 활용 |
|---|---|---|
| 클래스 균형 (Class Balancing) | Set A 에서 부족한 속성 클래스 (foggy / dawn-dusk 등) 를 채워 넣음 | ✅ 라벨로 직접 필터링 |
| Hard Example Mining | base 모델의 confidence 가 낮은 / 예측이 라벨과 다른 이미지를 우선 선택 | ✅ 모델 예측 vs 정답 비교 |
| 다양성 (Core-Set) | Set B 의 feature space 를 가장 잘 커버하는 부분집합 선택 (k-center / clustering) | 라벨 무관 |
| 결합 커버리지 | 속성 *조합* 의 균형을 맞춤 — 예: (snowy & night), (rainy & residential) | ✅ 라벨로 조합 카운트 |
| Loss 기반 | Set B 이미지에 대한 학습 직전 loss 가 큰 샘플 우선 | ✅ 라벨 필요 |

위 전략들을 결합/응용/대체할 수 있습니다. **Curation Report 에 본인의 의사결정 근거를 명확히 기술** 하세요.

**산출물**: `level5_picks.json` — 선택한 image_id 리스트 (이미지별 메타데이터 포함 가능).

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Lee-Yongsu/2026-HYU-AUE8088-PA2.git
else:
    !git reset --hard origin/main

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 2.00 KiB | 1023.00 KiB/s, done.
From https://github.com/Lee-Yongsu/2026-HYU-AUE8088-PA2
 * branch            main       -> FETCH_HEAD
   355a28d..d70ecfa  main       -> origin/main
Updating 0b69455..d70ecfa
error: Your local changes to the following files would be overwritten by merge:
	notebooks/level4_xai_efficiency.ipynb
Please commit your changes or stash them before you merge.
Aborting
/content/2026-HYU-AUE8088-PA2
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from collections import Counter

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.utils.submission import write_submission
from src.datasets.bdd_attr import (
    BDDAttrDataset, ATTRIBUTES, NUM_CLASSES, WEATHER_CLASSES, TIMEOFDAY_CLASSES,
)
from src.models.vit import vit_small_patch16_224, load_pretrained_vit_s16

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"            # 비활성화하려면 None
STRATEGY_NAME = "deficit-hard-dedup"    # 결손할당 × 하드샘플 + 중복제거
WANDB_TAGS    = ["level5", STRATEGY_NAME]

In [ ]:
# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨
DATA_ROOT = "../data/set_a"

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------


In [ ]:
# 1단계 — best 모델(Level3 ViT-S/16)로 Set B 전체를 1회 추론.
#   한 번의 forward 로 3가지를 동시에 수집한다:
#     (a) softmax 확률   → 불확실성(uncertainty)
#     (b) 정답 라벨       → 예측오류(error)        ※ Set B 라벨은 공개됨
#     (c) CLS 임베딩      → 중복제거(다양성)용 feature
#
# 주의: checkpoints/ 는 repo '내부'에 있음(데이터의 ../data 와 다름). cwd=repo 루트 기준 탐색.
CKPT_DIR  = next((d for d in ("checkpoints", "../checkpoints") if os.path.isdir(d)), "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
BEST_CKPT = os.path.join(CKPT_DIR, "level3_loss_sampler_mix.pth")
print("loading best model:", os.path.abspath(BEST_CKPT))

model = vit_small_patch16_224().to(device)
model.load_state_dict(torch.load(BEST_CKPT, map_location=device))   # raw state_dict
model.eval()

set_b = BDDAttrDataset("../data/set_b", split="mining", transform=eval_transform())
loader_b = DataLoader(set_b, batch_size=128, shuffle=False, num_workers=2)

# model.norm 출력의 0번째 토큰(CLS feature, 384-d)을 hook 으로 캡처
feats = []
h = model.norm.register_forward_hook(lambda m, i, o: feats.append(o[:, 0].detach().cpu()))
preds_b, probs_b, labels_b, ids_b = collect_predictions(model, loader_b, device)
h.remove()
feats_b = torch.cat(feats).numpy()                       # (N, 384)

# (a) 불확실성 = 1 - (세 head 의 max-softmax 평균)
max_probs   = np.stack([probs_b[a].max(axis=-1) for a in ATTRIBUTES], axis=1)   # (N, 3)
uncertainty = 1.0 - max_probs.mean(axis=1)                                       # (N,)

# (b) 예측오류 = 세 속성 중 틀린 비율 (labels_b = Set B 공개 정답)
error = np.mean([(preds_b[a] != labels_b[a]).astype(np.float32) for a in ATTRIBUTES], axis=0)  # (N,)

print(f"Set B: {len(ids_b)} images | feat={feats_b.shape} | "
      f"unc μ={uncertainty.mean():.3f} | err μ={error.mean():.3f}")

In [ ]:
# 2단계 — 선별 알고리즘 (무거운 버전): 결손할당 × 하드샘플 + 중복제거.
#
#   value_i = norm(rarity_i) · ( λ·norm(unc_i) + (1-λ)·err_i )
#     - rarity : Set A 의 (weather,scene,timeofday) 조합 역빈도  → "어떤 칸을" (결손)
#     - unc/err: base 모델 불확실성·예측오류                     → "그 칸의 누구를" (하드샘플)
#   greedy : value 내림차순으로 훑되,
#     - 조합 버킷 quota(=water-filling 목표선까지)를 넘기지 않고 (분포 쏠림 방지)
#     - 이미 고른 것과 CLS 코사인 유사도가 τ 초과면 skip          → 중복제거(다양성)
import json

K          = 1000
LAMBDA     = 0.5     # 불확실성 vs 예측오류 비중
SIM_TAU    = 0.92    # 코사인 유사도 이 값 초과 → 중복으로 보고 skip
TARGET_PCT = 75      # Set A 조합 분포의 이 분위수까지 각 버킷을 채움(목표선)

# --- 결손(deficit): Set A train 의 조합별 개수 (라벨만 필요 → transform 없이 로드) ---
train_ds = BDDAttrDataset("../data/set_a", "train")
def combo(w, s, t): return (int(w), int(s), int(t))
countA = Counter(combo(s.weather, s.scene, s.timeofday) for s in train_ds.samples)
target = int(np.percentile(list(countA.values()), TARGET_PCT))      # water-filling 목표선
def deficit(c): return max(0, target - countA.get(c, 0))            # 목표선까지 모자란 양

# --- 후보별 rarity + 결합 value ---
combos_b = [combo(labels_b["weather"][i], labels_b["scene"][i], labels_b["timeofday"][i])
            for i in range(len(ids_b))]
rarity = np.array([1.0 / (countA.get(c, 0) + 1.0) for c in combos_b])
def _norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-8)
value = _norm(rarity) * (LAMBDA * _norm(uncertainty) + (1 - LAMBDA) * error + 1e-6)

# --- greedy 선택: 버킷 quota + 중복제거 ---
featN = feats_b / (np.linalg.norm(feats_b, axis=1, keepdims=True) + 1e-8)   # 코사인용 정규화
quota = {c: deficit(c) for c in set(combos_b)}
order = np.argsort(-value)
picked = []
def too_similar(i):
    if not picked:
        return False
    return float((featN[picked] @ featN[i]).max()) > SIM_TAU

# 1차 패스 — 결손 quota 를 지키며 채움
for i in order:
    if len(picked) >= K:
        break
    c = combos_b[i]
    if quota.get(c, 0) <= 0 or too_similar(i):
        continue
    picked.append(int(i)); quota[c] -= 1

# 2차 패스 — quota 를 다 채우고도 1000 미달이면 value 순으로 보충
if len(picked) < K:
    chosen = set(picked)
    for i in order:
        if len(picked) >= K:
            break
        i = int(i)
        if i in chosen or too_similar(i):
            continue
        picked.append(i); chosen.add(i)

print(f"selected {len(picked)} / {K}  (target line={target}, λ={LAMBDA}, τ={SIM_TAU})")

# --- level5_picks.json 저장 (정답 = Set B 공개 라벨) ---
picks = [{
    "image_id":    ids_b[i],
    "weather":     int(labels_b["weather"][i]),
    "scene":       int(labels_b["scene"][i]),
    "timeofday":   int(labels_b["timeofday"][i]),
    "uncertainty": round(float(uncertainty[i]), 4),
    "error":       round(float(error[i]), 4),
} for i in picked]

with open("../level5_picks.json", "w") as f:
    json.dump({
        "strategy": ("결손할당(조합 역빈도 water-filling) × 하드샘플(불확실성+예측오류) "
                     "+ CLS 임베딩 코사인 중복제거. value=norm(rarity)·(λ·unc+(1-λ)·err)."),
        "params": {"K": K, "lambda": LAMBDA, "sim_tau": SIM_TAU, "target_pct": TARGET_PCT},
        "num_picks": len(picks),
        "picks": picks,
    }, f, indent=2)
print("saved ../level5_picks.json")

In [ ]:
# picks 분포 시각화 — 큐레이션 의도(비어있던 칸을 채웠는지) 검증. Curation Report 용.
import matplotlib.pyplot as plt

H = np.zeros((NUM_CLASSES["weather"], NUM_CLASSES["timeofday"]), dtype=int)
for p in picks:
    H[p["weather"], p["timeofday"]] += 1

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(H, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES["timeofday"])); ax.set_xticklabels(TIMEOFDAY_CLASSES, rotation=30, ha="right")
ax.set_yticks(range(NUM_CLASSES["weather"]));   ax.set_yticklabels(WEATHER_CLASSES)
for (r, c), v in np.ndenumerate(H):
    ax.text(c, r, v, ha="center", va="center", fontsize=9)
ax.set_title(f"picks: weather x timeofday  (n={len(picks)})")
ax.set_xlabel("timeofday"); ax.set_ylabel("weather")
fig.colorbar(im, fraction=0.046); fig.tight_layout()

In [ ]:
# 3단계 — 재학습 공통 함수. picks(또는 random)만 바꿔 '동일 레시피'로 학습 → 공정한 DI 비교.
#   base   = ImageNet 사전학습 ViT-S/16 (Level2/3 와 동일), 손실 = CE.
#   ※ 의도적으로 focal/CB/sampler 같은 불균형 보정 기법을 뺀다. 데이터 큐레이션 자체가
#     데이터-레벨 불균형 보정이므로, 손실/샘플러 트릭과 겹치면 DI 신호가 가려진다.
#     두 run(student / random)이 같은 레시피라 비교는 공정하다.
DEIT_URL  = "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth"
DEIT_PATH = os.path.join(CKPT_DIR, "deit_small_patch16_224.pth")   # CKPT_DIR = score-set-b 에서 해석됨
EPOCHS    = 25

val_ds     = BDDAttrDataset("../data/set_a", "val", transform=eval_transform())
loader_val = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

def build_pretrained_vit():
    set_seed(SEED, deterministic=True)
    m = vit_small_patch16_224().to(device)
    if not os.path.exists(DEIT_PATH):
        os.makedirs(os.path.dirname(DEIT_PATH), exist_ok=True)
        torch.hub.download_url_to_file(DEIT_URL, DEIT_PATH)
    load_pretrained_vit_s16(m, torch.load(DEIT_PATH, map_location="cpu"))
    return m

def run_retrain(extra_picks, run_name, tags, save_path=None):
    """Set A + extra_picks 로 재학습 → 최종 val Avg-MF1 과 모델을 반환."""
    train_aug = BDDAttrDataset("../data/set_a", "train",
                               transform=train_transform(), extra_picks=extra_picks)
    g = torch.Generator(); g.manual_seed(SEED)
    loader_tr = DataLoader(train_aug, batch_size=64, shuffle=True, num_workers=2,
                           worker_init_fn=seed_worker, generator=g, pin_memory=True)

    m = build_pretrained_vit()
    optim = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
    losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

    logger = WandbLogger(
        project=WANDB_PROJECT, run_name=run_name, tags=tags,
        config={"backbone": "vit_s16_pretrained", "strategy": run_name,
                "num_extra": len(extra_picks), "epochs": EPOCHS, "batch": 64,
                "lr": 1e-4, "weight_decay": 5e-2, "seed": SEED},
    )
    trainer = MultiTaskTrainer(m, optim, sched, losses, device,
                               TrainConfig(epochs=EPOCHS, lr=1e-4, weight_decay=5e-2), logger=logger)
    history = trainer.fit(loader_tr, loader_val)

    # 종료 후 — 속성별 confusion matrix 업로드
    vp, _, vt, _ = collect_predictions(m, loader_val, device)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(vp, vt)[a], CLASS_NAMES[a])
    if save_path:
        torch.save(m.state_dict(), save_path)
    logger.finish()
    return history["val_avg_mf1"][-1], m

# --- 내 picks 로 재학습 ---
extra = [(p["image_id"], p["weather"], p["scene"], p["timeofday"]) for p in picks]
mf1_student, model2 = run_retrain(
    extra, f"level5-{STRATEGY_NAME}", WANDB_TAGS, save_path=os.path.join(CKPT_DIR, "level5_final.pth"))
print(f"[student picks] val Avg-MF1 = {mf1_student:.4f}")

In [ ]:
# 4단계 — Random-1000 baseline. '동일 레시피'로 무작위 1000장 재학습 → DI 의 분모.
rng = np.random.default_rng(SEED)
rand_idx = rng.permutation(len(ids_b))[:K]
rand_extra = [(ids_b[i], int(labels_b["weather"][i]), int(labels_b["scene"][i]),
               int(labels_b["timeofday"][i])) for i in rand_idx]

mf1_random, _ = run_retrain(rand_extra, "level5-random-baseline", ["level5", "random"])
DI = (mf1_student - mf1_random) / mf1_random
print(f"[random ] val Avg-MF1 = {mf1_random:.4f}")
print(f"[student] val Avg-MF1 = {mf1_student:.4f}")
print(f"==> DI score = {DI:+.4f}  ({DI*100:+.2f}%)")

In [ ]:
# 5단계 — 추가 데이터의 '한계 효용' ablation: 우선순위 상위 250/500/1000 장으로 각각 재학습.
#   extra 는 greedy 선택 순서(=value 우선순위)이므로 extra[:n] = 가장 가치 큰 n 장.
abl_mf1 = {1000: mf1_student}            # 1000 은 위 student run 결과 재사용
for n in [250, 500]:
    mf1_n, _ = run_retrain(extra[:n], f"level5-ablation-{n}", ["level5", "ablation", f"n{n}"])
    abl_mf1[n] = mf1_n
    print(f"  picks={n:4d}  val Avg-MF1 = {mf1_n:.4f}")

import matplotlib.pyplot as plt
ns = [250, 500, 1000]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ns, [abl_mf1[n] for n in ns], "o-", label="student picks")
ax.axhline(mf1_random, ls="--", color="gray", label="random-1000")
ax.set_xlabel("# added images"); ax.set_ylabel("val Avg-MF1")
ax.set_title("Marginal utility of curated data"); ax.legend()
fig.tight_layout()

In [ ]:
# 6단계 — Kaggle 제출용 CSV 생성 (student picks 로 학습한 model2 사용).
test_ds = BDDAttrDataset("../data/set_a", "test", transform=eval_transform())
loader_te = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

preds_te, _, _, ids_te = collect_predictions(model2, loader_te, device)
write_submission("../submission/level5_submission.csv", ids_te, preds_te)
print("submission/level5_submission.csv 생성 완료 — Kaggle 페이지에 직접 업로드 하세요.")

## Curation Report — 필수

Final PPT 에 다음을 포함하세요.
- **선별 알고리즘** (의사코드 또는 1페이지 다이어그램).
- 본인 picks 1,000장의 **분포** — (예측된) weather × scene × timeofday — 를 heatmap 또는 stacked bar 로 시각화.
- **Random-1000 baseline** 결과와 본인의 **DI score** 비교.
- **Ablation**: 250 / 500 / 1000 장을 골랐을 때의 변화 — 추가 데이터의 한계 효용이 보이는지 확인.

여러 전략을 시험했다면 wandb 의 같은 프로젝트에 `STRATEGY_NAME` 만 바꿔서 별도 Run 으로 누적하세요. 학습 곡선·분포·DI score 비교가 한 페이지에 모입니다.